# 🧠 Challenge 18: AI 시맨틱 캐시 파이프라인

**난이도:** ⭐⭐⭐⭐
**사전 요구사항:** [02. Redis Deep Dive](./02_redis_deep_dive.ipynb), [04. Kafka Deep Dive](./04_kafka_deep_dive.ipynb)

---

## 시나리오

이커머스 고객지원 챗봇 시스템을 구축합니다.
고객 질문이 들어오면 **시맨틱 유사도 검색**으로 기존 Q&A에서 답변을 찾고,
못 찾으면 AI 추론 파이프라인을 통해 답변을 생성합니다.

**요구사항:**
- 30건의 기존 Q&A 데이터를 Redis Vector Set에 저장
- 10건의 테스트 질문으로 유사도 검색 정확도 확인
- 캐시 미스 시 Kafka로 추론 요청 스트리밍
- RabbitMQ로 답변 완료 알림 전송

**핵심 학습 포인트:**
- Redis 8 Vector Set (`VADD` / `VSIM`) 네이티브 벡터 유사도 검색
- 세 브로커(Redis + Kafka + RabbitMQ)를 하나의 AI 파이프라인에서 통합 활용
- 시맨틱 캐시로 추론 비용/지연 절감


## 아키텍처


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 360" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">AI 시맨틱 캐시(Semantic Cache) 아키텍처</text> <g transform="translate(250, 50)"> <rect x="0" y="0" width="300" height="40" rx="6" fill="#F1F5F9" stroke="#475569" stroke-width="1.5" /> <text x="150" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#1E293B" text-anchor="middle">고객 질문: "카드로 결제했는데 오류가 나요"</text> </g> <path d="M 400 90 L 400 120" stroke="#475569" stroke-width="2" /> <text x="410" y="110" font-family="-apple-system, sans-serif" font-size="10" fill="#475569">[1] 텍스트 → 벡터 변환</text> <g transform="translate(240, 130)"> <rect x="0" y="0" width="320" height="50" rx="8" fill="#E0F2FE" stroke="#0284C7" stroke-width="1.5" /> <text x="160" y="22" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#0369A1" text-anchor="middle">Redis Vector Set (VSIM) 검색</text> <text x="160" y="38" font-family="-apple-system, sans-serif" font-size="10" fill="#0C4A6E" text-anchor="middle">코사인 유사도로 가장 비슷한 캐시 질문 탐색</text> </g> <path d="M 320 180 L 190 220" stroke="#22C55E" stroke-width="2" /> <text x="215" y="195" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#15803D" text-anchor="middle">유사도 ≥ 0.8 (HIT)</text> <path d="M 480 180 L 610 220" stroke="#EF4444" stroke-width="2" /> <text x="585" y="195" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#B91C1C" text-anchor="middle">유사도 &lt; 0.8 (MISS)</text> <g transform="translate(40, 230)"> <rect x="0" y="0" width="280" height="90" rx="8" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="140" y="25" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">[2] Redis KV 캐시 조회</text> <text x="140" y="45" font-family="-apple-system, sans-serif" font-size="10" fill="#14532D" text-anchor="middle">캐시된 유사 질문의 답변 즉시 조회</text> <line x1="20" y1="55" x2="260" y2="55" stroke="#BBF7D0" stroke-width="1" /> <text x="140" y="75" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#166534" text-anchor="middle">반환 완료 (마이크로초 속도! ⚡)</text> </g> <g transform="translate(480, 230)"> <rect x="0" y="0" width="280" height="95" rx="8" fill="#FEF3C7" stroke="#D97706" stroke-width="1.5" /> <text x="140" y="20" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#B45309" text-anchor="middle">[3] Kafka 추론 요청 발행</text> <text x="140" y="38" font-family="-apple-system, sans-serif" font-size="9" fill="#78350F" text-anchor="middle">→ LLM 모델 추론 대기열 큐잉</text> <text x="140" y="55" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#B45309" text-anchor="middle">[4] Redis 캐시 저장 &amp; Vector Set 등록</text> <text x="140" y="72" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#7E22CE" text-anchor="middle">[5] RabbitMQ로 완료 알림 전송</text> <text x="140" y="87" font-family="-apple-system, sans-serif" font-size="8" fill="#701A75" text-anchor="middle">(사용자 브라우저로 실시간 알림 완료)</text> </g> </svg>
</div>



**브로커별 역할:**
| 브로커 | 역할 | 왜 이 브로커인가? |
|--------|------|----------------|
| **Redis** Vector Set | 벡터 유사도 검색 + 답변 캐시 | 마이크로초 응답, 인메모리 벡터 연산 |
| **Kafka** | 추론 요청 스트리밍 | 대량 요청 버퍼링, 리플레이 가능 |
| **RabbitMQ** | 답변 완료 알림 | 1:1 직접 전달, 사용자별 큐 |

## 환경 설정 및 데이터 로드

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import httpx
import json
import asyncio
import struct
import hashlib
import time
import numpy as np
import redis.asyncio as aioredis
from datetime import datetime

BASE_URL = "http://localhost:8000"
r = aioredis.from_url("redis://localhost:6379", decode_responses=False)
r_text = aioredis.from_url("redis://localhost:6379", decode_responses=True)

# Q&A 데이터 로드
data_path = Path("../data/mock/support_questions.json")
with open(data_path, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

questions = qa_data["questions"]
test_queries = qa_data["test_queries"]

print(f"📊 데이터 로드 완료")
print(f"  - 기존 Q&A: {len(questions)}건")
print(f"  - 테스트 질문: {len(test_queries)}건")
print(f"  - 카테고리: {', '.join(qa_data['metadata']['categories'])}")

## Step 1: Mock 임베딩 함수

**왜 Mock 임베딩인가?**

실제 AI 시스템에서는 OpenAI Embedding, Sentence-BERT 같은 모델을 사용합니다.
이 노트북에서는 **메시지 브로커 패턴 학습**이 목적이므로,
간단한 해시 기반 임베딩으로 벡터 검색 파이프라인을 시연합니다.


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 160" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">임베딩 흐름 비교: 프로덕션 vs 학습용</text> <g transform="translate(50, 50)"> <rect x="0" y="0" width="320" height="90" rx="8" fill="#FFFFFF" stroke="#0284C7" stroke-width="1.5" /> <text x="160" y="22" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#0369A1" text-anchor="middle">프로덕션 (Production)</text> <text x="160" y="45" font-family="-apple-system, sans-serif" font-size="11" fill="#334155" text-anchor="middle">텍스트 → OpenAI API → 768/1536차원 벡터</text> <text x="160" y="68" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#0284C7" text-anchor="middle">Redis VSIM 유사도 검색</text> </g> <g transform="translate(430, 50)"> <rect x="0" y="0" width="320" height="90" rx="8" fill="#F1F5F9" stroke="#64748B" stroke-width="1.5" /> <text x="160" y="22" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#475569" text-anchor="middle">이 노트북 (Message Broker Lab)</text> <text x="160" y="45" font-family="-apple-system, sans-serif" font-size="11" fill="#334155" text-anchor="middle">텍스트 → 해시함수(Mock) → 128차원 벡터</text> <text x="160" y="68" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#475569" text-anchor="middle">동일한 Redis VSIM 유사도 검색</text> </g> </svg>
</div>



In [ ]:
EMBEDDING_DIM = 128

def text_to_embedding(text: str) -> np.ndarray:
    """텍스트에서 결정론적 임베딩 생성 (학습용 Mock)"""
    # 한글 키워드 기반 특징 추출: 유사한 의미의 텍스트가 비슷한 벡터를 생성하도록
    keywords = {
        "결제": 0, "카드": 1, "환불": 2, "취소": 3, "할부": 4, "포인트": 5,
        "배송": 6, "택배": 7, "배달": 8, "추적": 9, "조회": 10,
        "반품": 11, "교환": 12, "환불": 13, "반송": 14,
        "회원": 15, "비밀번호": 16, "로그인": 17, "탈퇴": 18, "등급": 19,
        "상품": 20, "사이즈": 21, "품절": 22, "리뷰": 23, "선물": 24,
        "오류": 25, "실패": 26, "안된다": 27, "안돼": 27, "안돼요": 27,
        "영수증": 28, "쿠폰": 29, "적립금": 30, "새벽": 31,
        "해외": 32, "가격": 33, "할인": 34, "주문": 35
    }
    
    # 기본 벡터: 텍스트 해시 기반
    seed = int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)
    rng = np.random.RandomState(seed)
    vec = rng.randn(EMBEDDING_DIM).astype(np.float32)
    
    # 키워드 매칭: 특정 차원에 가중치 부여
    for kw, idx in keywords.items():
        if kw in text:
            vec[idx % EMBEDDING_DIM] += 3.0
    
    # L2 정규화
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    
    return vec


def embedding_to_bytes(vec: np.ndarray) -> bytes:
    """numpy 배열을 FP32 바이트로 변환 (Redis VADD/VSIM 용)"""
    return vec.astype(np.float32).tobytes()


# 테스트: 유사한 질문이 비슷한 벡터를 생성하는지 확인
v1 = text_to_embedding("카드 결제가 안 돼요")
v2 = text_to_embedding("카드로 결제했는데 오류가 나요")
v3 = text_to_embedding("택배 배송 조회하고 싶어요")

sim_12 = np.dot(v1, v2)
sim_13 = np.dot(v1, v3)

print(f"🔍 임베딩 유사도 테스트")
print(f"  '카드 결제가 안 돼요' vs '카드로 결제했는데 오류가 나요': {sim_12:.4f} (↑ 높아야 정상)")
print(f"  '카드 결제가 안 돼요' vs '택배 배송 조회하고 싶어요': {sim_13:.4f} (↓ 낮아야 정상)")

## Step 2: Redis Vector Set에 Q&A 벡터 저장 (VADD)

Redis 8의 **Vector Set**은 벡터 데이터를 네이티브로 저장하고 유사도 검색을 수행합니다.

```text
VADD key REDUCE dim FP32 <vector_bytes> element_name

- key: Vector Set 이름
- REDUCE dim: 벡터를 dim 차원으로 압축 저장 (메모리 절약)
- FP32: 32비트 부동소수점 형식
- element_name: 검색 결과로 반환될 이름 (Q&A ID)
```

In [ ]:
VECTOR_KEY = "support:vectors"
CACHE_PREFIX = "support:answer:"

async def build_vector_index():
    """기존 Q&A를 Vector Set에 저장"""
    
    # 기존 데이터 정리
    await r.delete(VECTOR_KEY)
    
    success = 0
    for qa in questions:
        vec = text_to_embedding(qa["question"])
        vec_bytes = embedding_to_bytes(vec)
        
        try:
            # VADD: Vector Set에 벡터 추가
            await r.execute_command(
                "VADD", VECTOR_KEY,
                "REDUCE", EMBEDDING_DIM,
                "FP32", vec_bytes,
                qa["id"]
            )
            
            # 답변을 Redis KV에 캐시
            cache_data = json.dumps({
                "question": qa["question"],
                "answer": qa["answer"],
                "category": qa["category"]
            }, ensure_ascii=False)
            await r_text.set(f"{CACHE_PREFIX}{qa['id']}", cache_data)
            
            success += 1
        except Exception as e:
            if "VADD" in str(e) or "unknown command" in str(e).lower():
                print(f"\u26a0\ufe0f  Redis Vector Set이 지원되지 않습니다. Redis 8.4+ 버전이 필요합니다.")
                print(f"   현재 Redis 버전을 확인하세요: redis-cli INFO server | grep redis_version")
                return False
            raise
    
    # Vector Set 정보 확인
    info = await r.execute_command("VCARD", VECTOR_KEY)
    print(f"\u2705 Vector Set 구축 완료")
    print(f"  - 저장된 벡터: {info}개")
    print(f"  - 차원: {EMBEDDING_DIM}")
    print(f"  - Q&A 캐시: {success}건")
    return True

vector_ready = await build_vector_index()

## Step 3: 유사도 검색 (VSIM) + 시맨틱 캐시

**VSIM** 명령어로 쿼리 벡터와 가장 유사한 Q&A를 찾습니다.
유사도가 임계값(≥ 0.7) 이상이면 **캐시 HIT**, 아니면 **캐시 MISS**입니다.

```text
VSIM key FP32 <query_vector> COUNT 3 WITHSCORES

반환값: [element1, score1, element2, score2, ...]
score = 코사인 유사도 (0~1, 1이 가장 유사)
```

In [ ]:
SIMILARITY_THRESHOLD = 0.7

async def semantic_search(query: str, top_k: int = 3):
    """시맨틱 유사도 검색 및 캐시 확인"""
    query_vec = text_to_embedding(query)
    query_bytes = embedding_to_bytes(query_vec)
    
    # VSIM: 유사한 벡터 검색
    results = await r.execute_command(
        "VSIM", VECTOR_KEY,
        "FP32", query_bytes,
        "COUNT", top_k,
        "WITHSCORES"
    )
    
    # 결과 파싱: [id, score, id, score, ...]
    matches = []
    for i in range(0, len(results), 2):
        qa_id = results[i].decode() if isinstance(results[i], bytes) else results[i]
        score = float(results[i + 1])
        matches.append({"id": qa_id, "score": score})
    
    return matches


async def get_cached_answer(qa_id: str):
    """Redis KV에서 캐시된 답변 조회"""
    data = await r_text.get(f"{CACHE_PREFIX}{qa_id}")
    if data:
        return json.loads(data)
    return None


# 테스트 검색
if vector_ready:
    test_q = "카드로 결제했는데 오류가 나요"
    matches = await semantic_search(test_q)
    
    print(f"\ud83d\udd0d 검색 쿼리: '{test_q}'")
    print(f"\n\ud83c\udfaf Top-3 유사 질문:")
    for m in matches:
        answer = await get_cached_answer(m['id'])
        q_text = answer['question'] if answer else '?'
        status = "\u2705 HIT" if m['score'] >= SIMILARITY_THRESHOLD else "\u274c MISS"
        print(f"  [{status}] {m['id']} (\uc720\uc0ac\ub3c4: {m['score']:.4f})")
        print(f"        \uc9c8\ubb38: {q_text}")
        if answer and m['score'] >= SIMILARITY_THRESHOLD:
            print(f"        \ub2f5\ubcc0: {answer['answer'][:60]}...")

## Step 4: 전체 파이프라인 — 캐시 HIT/MISS 분기

**캐시 HIT**: Redis에서 즉시 답변 반환 (마이크로초)
**캐시 MISS**: Kafka에 추론 요청 → 시뮬레이션 → 결과 캐싱 + RabbitMQ 알림

In [ ]:
async def handle_customer_query(query: str, session_id: str):
    """\uace0\uac1d \uc9c8\ubb38 \ucc98\ub9ac \ud30c\uc774\ud504\ub77c\uc778"""
    start = time.time()
    
    # 1) 유사도 검색
    matches = await semantic_search(query, top_k=1)
    best = matches[0] if matches else None
    
    async with httpx.AsyncClient(base_url=BASE_URL, timeout=10.0) as client:
        if best and best["score"] >= SIMILARITY_THRESHOLD:
            # === CACHE HIT ===
            answer_data = await get_cached_answer(best["id"])
            elapsed = (time.time() - start) * 1000
            
            return {
                "status": "CACHE_HIT",
                "session_id": session_id,
                "query": query,
                "matched_id": best["id"],
                "similarity": best["score"],
                "answer": answer_data["answer"] if answer_data else None,
                "latency_ms": round(elapsed, 2)
            }
        else:
            # === CACHE MISS: Kafka에 추론 요청 발행 ===
            inference_request = json.dumps({
                "session_id": session_id,
                "query": query,
                "timestamp": datetime.now().isoformat()
            }, ensure_ascii=False)
            
            await client.post("/kafka/basic/publish", json={
                "content": inference_request,
                "metadata": {"type": "inference_request", "session": session_id}
            })
            
            # 추론 시뮬레이션 (\uc2e4\uc81c\ub85c\ub294 ML \ubaa8\ub378 \ud638\ucd9c)
            simulated_answer = f"[AI \uc0dd\uc131] '{query}'\uc5d0 \ub300\ud55c \ub2f5\ubcc0: \uace0\uac1d\uc9c0\uc6d0\ud300\uc5d0\uc11c \ud655\uc778 \ud6c4 \uc548\ub0b4\ub4dc\ub9ac\uaca0\uc2b5\ub2c8\ub2e4."
            
            # 결과를 Redis에 캐싱
            new_id = f"ai_{session_id}"
            cache_data = json.dumps({
                "question": query,
                "answer": simulated_answer,
                "category": "ai_generated"
            }, ensure_ascii=False)
            await r_text.set(f"{CACHE_PREFIX}{new_id}", cache_data, ex=3600)
            
            # Vector Set에 새 벡터 추가 (다음 유사 질문은 캐시 HIT)
            vec = text_to_embedding(query)
            await r.execute_command(
                "VADD", VECTOR_KEY,
                "REDUCE", EMBEDDING_DIM,
                "FP32", embedding_to_bytes(vec),
                new_id
            )
            
            # RabbitMQ로 답변 완료 알림
            await client.post(
                "/rabbitmq/direct/publish",
                params={"queue_name": "support-notifications"},
                json={
                    "content": f"\ub2f5\ubcc0 \uc644\ub8cc: {session_id}",
                    "metadata": {"session_id": session_id, "type": "answer_ready"}
                }
            )
            
            elapsed = (time.time() - start) * 1000
            
            return {
                "status": "CACHE_MISS",
                "session_id": session_id,
                "query": query,
                "similarity": best["score"] if best else 0,
                "answer": simulated_answer,
                "latency_ms": round(elapsed, 2),
                "cached_as": new_id
            }

print("\u2705 \ud30c\uc774\ud504\ub77c\uc778 \ud568\uc218 \uc815\uc758 \uc644\ub8cc")

## Step 5: 테스트 실행 — 10건 질문 처리

유사 질문(\uce90\uc2dc HIT 예\uc0c1)과 \uc0c8\ub85c\uc6b4 \uc9c8\ubb38(\uce90\uc2dc MISS \uc608\uc0c1)\uc744 \uc12c\uc5b4\uc11c \ud14c\uc2a4\ud2b8\ud569\ub2c8\ub2e4.

In [ ]:
if vector_ready:
    print("\ud83d\ude80 \uace0\uac1d\uc9c0\uc6d0 \uc2dc\ub9e8\ud2f1 \uce90\uc2dc \ud14c\uc2a4\ud2b8\n")
    print("=" * 70)
    
    results = []
    for i, tq in enumerate(test_queries):
        session_id = f"sess_{i+1:03d}"
        result = await handle_customer_query(tq["query"], session_id)
        results.append(result)
        
        icon = "\u2705" if result["status"] == "CACHE_HIT" else "\ud83d\udce1"
        print(f"{icon} [{result['status']:10s}] '{tq['query'][:30]}'")
        print(f"   \uc720\uc0ac\ub3c4: {result['similarity']:.4f} | \uc9c0\uc5f0: {result['latency_ms']:.1f}ms")
    
    print("\n" + "=" * 70)
    
    # \ud1b5\uacc4
    hits = [r for r in results if r["status"] == "CACHE_HIT"]
    misses = [r for r in results if r["status"] == "CACHE_MISS"]
    
    print(f"\n\ud83d\udcca \uacb0\uacfc \ud1b5\uacc4:")
    print(f"  - \uce90\uc2dc HIT:  {len(hits)}/{len(results)}\uac74")
    print(f"  - \uce90\uc2dc MISS: {len(misses)}/{len(results)}\uac74")
    print(f"  - \ud788\ud2b8\uc728: {len(hits)/len(results)*100:.1f}%")
    
    if hits:
        avg_hit = sum(r["latency_ms"] for r in hits) / len(hits)
        print(f"  - \ud3c9\uade0 HIT \uc9c0\uc5f0: {avg_hit:.1f}ms")
    if misses:
        avg_miss = sum(r["latency_ms"] for r in misses) / len(misses)
        print(f"  - \ud3c9\uade0 MISS \uc9c0\uc5f0: {avg_miss:.1f}ms")
    
    if hits and misses:
        speedup = (sum(r["latency_ms"] for r in misses) / len(misses)) / (sum(r["latency_ms"] for r in hits) / len(hits))
        print(f"\n  \u26a1 \uce90\uc2dc HIT\uac00 MISS \ub300\ube44 {speedup:.1f}\ubc30 \ube60\ub984")

## Step 6: 캐시 워밍업 효과 확인

Step 5에서 MISS된 질문이 Vector Set에 추가되었으므로,
**동일한 질문을 다시 보내면 캐시 HIT**가 되어야 합니다.

In [ ]:
if vector_ready:
    print("\ud83d\udd01 \uce90\uc2dc \uc6cc\ubc0d\uc5c5 \ud14c\uc2a4\ud2b8 (\ub3d9\uc77c \uc9c8\ubb38 \uc7ac\uc804\uc1a1)\n")
    
    warmup_results = []
    for i, tq in enumerate(test_queries):
        session_id = f"warmup_{i+1:03d}"
        result = await handle_customer_query(tq["query"], session_id)
        warmup_results.append(result)
        
        icon = "\u2705" if result["status"] == "CACHE_HIT" else "\u274c"
        print(f"  {icon} [{result['status']:10s}] '{tq['query'][:35]}' ({result['latency_ms']:.1f}ms)")
    
    hits_after = len([r for r in warmup_results if r["status"] == "CACHE_HIT"])
    print(f"\n\ud83c\udfaf \uce90\uc2dc \ud788\ud2b8\uc728: {hits_after}/{len(warmup_results)} ({hits_after/len(warmup_results)*100:.0f}%)")
    print(f"\ud83d\udca1 \ud559\uc2b5 \ud6a8\uacfc: \uccab \uc694\uccad\uc5d0\uc11c MISS\ub41c \uc9c8\ubb38\ub3c4 \ub450 \ubc88\uc9f8\ubd80\ud130\ub294 HIT\ub85c \ucc98\ub9ac\ub429\ub2c8\ub2e4.")

## 결과 검증

In [ ]:
if vector_ready:
    print("\ud83d\udd0d \uc2dc\uc2a4\ud15c \uac80\uc99d\n")
    
    # 1. Vector Set \ud06c\uae30 \ud655\uc778
    vcard = await r.execute_command("VCARD", VECTOR_KEY)
    print(f"  \u2713 Vector Set \ud06c\uae30: {vcard}\uac1c (\ucd08\uae30 30 + \uc0c8\ub85c \ucd94\uac00\ub41c \uac83)")
    
    # 2. \uce90\uc2dc \ub370\uc774\ud130 \ud655\uc778
    keys = await r_text.keys(f"{CACHE_PREFIX}*")
    print(f"  \u2713 \uce90\uc2dc\ub41c \ub2f5\ubcc0: {len(keys)}\uac74")
    
    # 3. Kafka \ubc1c\ud589 \ud655\uc778 (\ud1a0\ud53d \uc815\ubcf4)
    async with httpx.AsyncClient(base_url=BASE_URL, timeout=5.0) as client:
        resp = await client.get("/kafka/topics")
        topics = resp.json()
        print(f"  \u2713 Kafka \ud1a0\ud53d \uc218: {topics.get('topic_count', '?')}")
        
        # 4. RabbitMQ \uc54c\ub9bc \ud050 \ud655\uc778
        resp = await client.get("/rabbitmq/queue/info/support-notifications")
        queue_info = resp.json()
        print(f"  \u2713 RabbitMQ \uc54c\ub9bc \ud050: {queue_info}")
    
    print(f"\n\u2705 \uac80\uc99d \uc644\ub8cc!")
    print(f"\n\ud83c\udfaf \ud575\uc2ec \uc131\uacfc:")
    print(f"  1. Redis Vector Set\uc73c\ub85c {vcard}\uac74\uc758 Q&A \ubca1\ud130 \uc800\uc7a5 \ubc0f \uc720\uc0ac\ub3c4 \uac80\uc0c9")
    print(f"  2. Kafka\ub85c \uce90\uc2dc MISS \uc2dc \ucd94\ub860 \uc694\uccad \uc2a4\ud2b8\ub9ac\ubc0d")
    print(f"  3. RabbitMQ\ub85c \ub2f5\ubcc0 \uc644\ub8cc \uc54c\ub9bc \uc804\uc1a1")
    print(f"  4. \uc2dc\ub9e8\ud2f1 \uce90\uc2dc \uc6cc\ubc0d\uc5c5\uc73c\ub85c \ud788\ud2b8\uc728 \uc810\uc9c4\uc801 \ud5a5\uc0c1")

## 핵심 포인트

### 시맨틱 캐시란?

일반적인 캐시는 **정확히 똑같은 입력**일 때만 HIT됩니다.
시맨틱 캐시는 **의미가 비슷한 입력**이면 HIT됩니다.

```text
[일반 캐시]                        [시맨틱 캐시]

"카드 결제가 안 돼요"  → HIT        "카드 결제가 안 돼요"  → HIT
"카드 결제 오류"      → MISS!      "카드 결제 오류"      → HIT! (의미가 비슷하므로)
"결제 안됨"           → MISS!      "결제 안됨"           → HIT! (의미가 비슷하므로)
```

### AI 시스템에서 메시지 브로커의 역할

| 구간 | 일반적 방법 | 브로커 활용 시 |
|------|------------|-------------|
| 추론 요청 | 동기 API 호출 | Kafka로 비동기 스트리밍 (베터 처리량, 리플레이) |
| 결과 캐싱 | 증해시 DB 직접 조회 | Redis Vector Set (마이크로초 응답) |
| 완료 알림 | 폴링/웹소켓 | RabbitMQ 직접 전달 (사용자별 큐) |
| 비용 절감 | 항상 AI 호출 | 시맨틱 캐시로 70~90% 호출 절감 |

### 프로덕션에서는?

```text
Mock Embedding → OpenAI text-embedding-3-small (1536차원)
                  또는 Sentence-BERT (384차원)
                  또는 Voyage AI, Cohere Embed 등

Redis VSIM     → 동일 (Redis 8 네이티브 벡터 검색)
                  또는 Pinecone, Qdrant, Weaviate 등 전용 벡터 DB

\uc2dc\ubbac\ub808\uc774\uc158 추\ub860 → GPT-4, Claude, Gemini \ub4f1 LLM API \ud638\ucd9c
```

## 확장 아이디어

### 1. RAG (Retrieval-Augmented Generation) 파이프라인
```python
# Vector Set에서 관련 문서 검색 → LLM에 컨텍스트로 전달
context_docs = await redis.vsim("docs:vectors", query_vec, count=5)
prompt = f"\ub2e4\uc74c \ubb38\uc11c\ub97c \ucc38\uace0\ud558\uc5ec \ub2f5\ubcc0:\n{context_docs}\n\n\uc9c8\ubb38: {query}"
answer = await llm.generate(prompt)
```

### 2. 실시간 임베딩 업데이트 파이프라인
```python
# 새 문서가 등록되면 Kafka로 임베딩 요청 스트리밍
# Consumer가 임베딩 생성 후 Redis Vector Set에 자동 추가
kafka_consumer.subscribe("new-documents")
for doc in kafka_consumer:
    embedding = model.encode(doc.content)
    redis.vadd("docs:vectors", embedding, doc.id)
```

### 3. A/B 테스트 with Kafka
```python
# 추론 요청을 파티션 키로 분배하여 A/B 맨 모델 비교
# 파티션 0,1 → Model A / 파티션 2 → Model B
kafka.publish_keyed("inference", key=session_id, message=request)
```

In [ ]:
# Cleanup
async def cleanup():
    """\ud14c\uc2a4\ud2b8 \ub370\uc774\ud130 \uc815\ub9ac"""
    print("\ud83e\uddf9 \uc815\ub9ac \uc911...\n")
    
    # Vector Set \uc0ad\uc81c
    await r.delete(VECTOR_KEY)
    print(f"  \u2713 Vector Set \uc0ad\uc81c: {VECTOR_KEY}")
    
    # \uce90\uc2dc \ub370\uc774\ud130 \uc0ad\uc81c
    keys = await r_text.keys(f"{CACHE_PREFIX}*")
    if keys:
        await r_text.delete(*keys)
    print(f"  \u2713 \uce90\uc2dc \uc0ad\uc81c: {len(keys)}\uac74")
    
    await r.close()
    await r_text.close()
    print(f"\n\u2705 \uc815\ub9ac \uc644\ub8cc!")

await cleanup()

## 네비게이션

- [← 이전: Challenge 17 - 이미지 파이프라인](./17_challenge_image_pipeline.ipynb)
- [↑ 목록: 프로젝트 개요](./01_project_overview.ipynb)